In [1]:
from pathlib import Path
from dotenv import load_dotenv
import json
import os
import re

import pandas as pd
import requests

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

ROOT = Path.cwd().resolve()

while not (ROOT / "_quarto.yml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent



RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_json_file = RAW_DIR / "data_scientist_naics5132_raw.json"
raw_csv_file = RAW_DIR / "data_scientist_naics5132_raw.csv"
processed_file = PROCESSED_DIR / "data_scientist_software_publishers.csv"


In [2]:
load_dotenv(ROOT / ".env", override=True)

API_KEY = os.getenv("EMPLOYABILITY_API_KEY", "").strip()
BASE_URL = os.getenv("EMPLOYABILITY_API_BASE_URL", "").strip().rstrip("/")
PATH_PREFIX = os.getenv("EMPLOYABILITY_API_PATH_PREFIX", "").strip().strip("/")

if not API_KEY:
    raise RuntimeError("EMPLOYABILITY_API_KEY is missing from .env.")

if not BASE_URL or not PATH_PREFIX:
    raise RuntimeError(
        "EMPLOYABILITY_API_BASE_URL or "
        "EMPLOYABILITY_API_PATH_PREFIX is missing from .env."
    )

API_URL = f"{BASE_URL}/{PATH_PREFIX}"

HEADERS = {
    "X-API-Key": API_KEY,
    "Accept": "application/json",
}

In [ ]:
import json
import time

RUN_TIME_SECONDS = 300
LIMIT = 100
SEARCH_QUERY = "data scientist"


data_scientist_json_file = (
    RAW_DIR / "data_scientist_query_naics5132_raw.json"
)

data_scientist_csv_file = (
    RAW_DIR / "data_scientist_query_naics5132_raw.csv"
)

if data_scientist_json_file.exists():
    with open(data_scientist_json_file, "r", encoding="utf-8") as file:
        saved_payload = json.load(file)

    all_rows = saved_payload.get("results", [])
else:
    all_rows = []

next_offset = len(all_rows)
start_time = time.monotonic()

while time.monotonic() - start_time < RUN_TIME_SECONDS:
    print(f"Requesting Data Scientist rows from offset {next_offset}...")

    response = requests.get(
        f"{API_URL}/jobs/",
        headers=HEADERS,
        params={
            "naics": "513210",
            "q": SEARCH_QUERY,
            "limit": LIMIT,
            "offset": next_offset,
        },
        timeout=180,
    )

    response.raise_for_status()

    page_payload = response.json()
    new_rows = page_payload.get("results", [])

    if not new_rows:
        print("No additional rows returned.")
        break

    all_rows.extend(new_rows)

    # Remove duplicate job IDs.
    unique_rows = []
    seen_job_ids = set()

    for row in all_rows:
        job_id = row.get("job_id")

        if job_id not in seen_job_ids:
            unique_rows.append(row)
            seen_job_ids.add(job_id)

    all_rows = unique_rows

    raw_payload = {
        "request": {
            "naics": "5132",
            "q": SEARCH_QUERY,
            "limit_per_page": LIMIT,
        },
        "results": all_rows,
    }

    with open(data_scientist_json_file, "w", encoding="utf-8") as file:
        json.dump(raw_payload, file, ensure_ascii=False, indent=2)

    raw_df = pd.json_normalize(all_rows)
    raw_df.to_csv(data_scientist_csv_file, index=False)

    print(f"Saved {len(raw_df)} total Data Scientist query rows.")

    if not page_payload.get("meta", {}).get("next"):
        print("Reached the final page.")
        break

    next_offset += LIMIT

print("Finished the Data Scientist pull.")
print("Total saved rows:", len(all_rows))

Requesting Data Scientist rows from offset 0...
No additional rows returned.
Finished the Data Scientist pull.
Total saved rows: 0


In [25]:
# Keep Data Scientist and closely related AI/ML roles

role_terms = [
    r"\bdata scientist\b",
    r"\bsenior data scientist\b",
    r"\bjunior data scientist\b",
    r"\blead data scientist\b",
    r"\bprincipal data scientist\b",
    r"\bmachine learning scientist\b",
    r"\bapplied scientist\b",
    r"\bresearch scientist\b",
    r"\bmachine learning engineer\b",
    r"\bml engineer\b",
    r"\bai engineer\b",
]

role_pattern = "|".join(role_terms)

raw_df["title_clean"] = (
    raw_df["title"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

raw_df["role_match"] = raw_df["title_clean"].str.contains(
    role_pattern,
    regex=True,
    na=False,
)

clean_df = raw_df[raw_df["role_match"]].copy()

print("Data Scientist and related-role matches:", len(clean_df))

display(
    clean_df[
        [
            "job_id",
            "title",
            "company_name",
            "naics_code",
            "naics_name",
        ]
    ]
)

Data Scientist and related-role matches: 9


,job_id,title,company_name,naics_code,naics_name
91,695086,AI Engineer (m/w/d) - LLM-Systeme & Softwareintegration,Macs Software GmbH,513200,Software Publishers
461,783636,Data scientist,Katholieke Universiteit Te Leuven AV,513200,Software Publishers
789,1490481,AI ENGINEER,Europa,513200,Software Publishers
901,1679378,Data Scientist / Lab Specialist m/w/d - Roche Penzberg (Data Scientist),Franz & Wach Personalservice PEN,513200,Software Publishers
1764,814504,Senior Software and AI Engineer,Coreweave,513200,Software Publishers
1871,817484,"Senior Software Engineer, Forward Deployed AI Engineer",Mongodb,513200,Software Publishers
1876,817485,"Senior Software Engineer, Forward Deployed AI Engineer",Mongodb,513200,Software Publishers
3004,972125,Senior Lead Software Engineer - ML Engineer for Agent Platform,JPMorgan Chase,513200,Software Publishers
3311,830950,"Job-30477 Software Architect / AI Engineer — SDLC Agêntico, Brazil",Ci And T,513200,Software Publishers


In [26]:
staffing_terms = [
    "staffing",
    "recruit",
    "recruiting",
    "talent",
    "employment",
    "personaldienst",
    "personalservice",
    "constaff",
    "akkodis",
    "orizon",
    "cure-x",
]

staffing_pattern = "|".join(
    re.escape(term)
    for term in staffing_terms
)

clean_df["company_clean"] = (
    clean_df["company_name"]
    .fillna("")
    .str.lower()
    .str.strip()
)

clean_df["company_name_staffing_flag"] = (
    clean_df["company_clean"]
    .str.contains(staffing_pattern, regex=True, na=False)
)

if "is_staffing_agency" in clean_df.columns:
    clean_df["api_staffing_flag"] = (
        clean_df["is_staffing_agency"]
        .fillna(False)
        .astype(bool)
    )
else:
    clean_df["api_staffing_flag"] = False

clean_df["staffing_flag"] = (
    clean_df["company_name_staffing_flag"]
    | clean_df["api_staffing_flag"]
)

clean_df = clean_df[~clean_df["staffing_flag"]].copy()

print("Rows after removing staffing agencies:", len(clean_df))

display(clean_df[["job_id", "title", "company_name"]])

Rows after removing staffing agencies: 8


,job_id,title,company_name
91,695086,AI Engineer (m/w/d) - LLM-Systeme & Softwareintegration,Macs Software GmbH
461,783636,Data scientist,Katholieke Universiteit Te Leuven AV
789,1490481,AI ENGINEER,Europa
1764,814504,Senior Software and AI Engineer,Coreweave
1871,817484,"Senior Software Engineer, Forward Deployed AI Engineer",Mongodb
1876,817485,"Senior Software Engineer, Forward Deployed AI Engineer",Mongodb
3004,972125,Senior Lead Software Engineer - ML Engineer for Agent Platform,JPMorgan Chase
3311,830950,"Job-30477 Software Architect / AI Engineer — SDLC Agêntico, Brazil",Ci And T


In [27]:
# Remove duplicates and clean date, salary, remote, and location fields

clean_df = clean_df.drop_duplicates(subset="job_id").copy()

for column in ["posted_at", "expires_at"]:
    if column in clean_df.columns:
        clean_df[column] = pd.to_datetime(
            clean_df[column],
            errors="coerce",
            utc=True,
        )

for column in [
    "annual_salary_min",
    "annual_salary_max",
    "hourly_salary_min",
    "hourly_salary_max",
]:
    if column in clean_df.columns:
        clean_df[column] = pd.to_numeric(
            clean_df[column],
            errors="coerce",
        )

if {
    "annual_salary_min",
    "annual_salary_max",
}.issubset(clean_df.columns):
    clean_df["annual_salary_midpoint"] = (
        clean_df["annual_salary_min"]
        + clean_df["annual_salary_max"]
    ) / 2

if "remote_status" in clean_df.columns:
    clean_df["remote_status"] = (
        clean_df["remote_status"]
        .fillna("unknown")
        .str.lower()
        .str.strip()
    )

for column in ["city", "state", "location_text"]:
    if column in clean_df.columns:
        clean_df[column] = (
            clean_df[column]
            .fillna("Unknown")
            .astype(str)
            .str.strip()
        )

In [28]:
# Convert skill lists into readable text

def format_skills(value):
    if not isinstance(value, list):
        return ""

    names = []

    for item in value:
        if isinstance(item, dict) and item.get("name"):
            names.append(str(item["name"]))
        elif isinstance(item, str):
            names.append(item)

    return ", ".join(names)

if "skills" in clean_df.columns:
    clean_df["skills_text"] = clean_df["skills"].apply(format_skills)
else:
    clean_df["skills_text"] = ""

In [29]:
helper_columns = [
    "company_clean",
    "company_name_staffing_flag",
    "api_staffing_flag",
]

clean_df = clean_df.drop(
    columns=[
        column
        for column in helper_columns
        if column in clean_df.columns
    ]
)

clean_df.to_csv(processed_file, index=False)

print("Final cleaned rows:", len(clean_df))
print("Saved cleaned dataset to:", processed_file)

display(
    clean_df[
        [
            column
            for column in [
                "job_id",
                "title",
                "company_name",
                "posted_at",
                "remote_status",
                "annual_salary_min",
                "annual_salary_max",
                "skills_text",
            ]
            if column in clean_df.columns
        ]
    ]
)

Final cleaned rows: 8
Saved cleaned dataset to: /home/ubuntu/ad688-career-evaluation-group9/data/processed/data_scientist_software_publishers.csv


,job_id,title,company_name,posted_at,remote_status,annual_salary_min,annual_salary_max,skills_text
91,695086,AI Engineer (m/w/d) - LLM-Systeme & Softwareintegration,Macs Software GmbH,2026-08-01 00:00:00+00:00,unknown,NaN,NaN,
461,783636,Data scientist,Katholieke Universiteit Te Leuven AV,NaT,unknown,NaN,NaN,Agentic AI
789,1490481,AI ENGINEER,Europa,NaT,unknown,NaN,NaN,
1764,814504,Senior Software and AI Engineer,Coreweave,2026-07-21 19:11:42+00:00,unknown,182000.0,242000.0,
1871,817484,"Senior Software Engineer, Forward Deployed AI Engineer",Mongodb,2026-07-21 14:26:00+00:00,unknown,0.0,0.0,
1876,817485,"Senior Software Engineer, Forward Deployed AI Engineer",Mongodb,2026-07-21 14:26:00+00:00,unknown,0.0,0.0,
3004,972125,Senior Lead Software Engineer - ML Engineer for Agent Platform,JPMorgan Chase,2026-07-20 00:00:00+00:00,unknown,NaN,NaN,
3311,830950,"Job-30477 Software Architect / AI Engineer — SDLC Agêntico, Brazil",Ci And T,2026-07-17 18:38:45+00:00,unknown,0.0,0.0,


In [30]:
data_dictionary = pd.DataFrame(
    [
        {
            "variable": "job_id",
            "description": "Unique API identifier for each job posting.",
        },
        {
            "variable": "title",
            "description": "Original job title from the posting.",
        },
        {
            "variable": "company_name",
            "description": "Employer name from the posting.",
        },
        {
            "variable": "posted_at",
            "description": "Posting date, when available.",
        },
        {
            "variable": "naics_code",
            "description": "API-provided NAICS industry code.",
        },
        {
            "variable": "naics_name",
            "description": "API-provided NAICS industry label.",
        },
        {
            "variable": "remote_status",
            "description": "Remote, hybrid, onsite, or unknown work arrangement.",
        },
        {
            "variable": "annual_salary_min",
            "description": "Normalized lower annual-salary amount.",
        },
        {
            "variable": "annual_salary_max",
            "description": "Normalized upper annual-salary amount.",
        },
        {
            "variable": "annual_salary_midpoint",
            "description": "Average of the annual minimum and maximum salary.",
        },
        {
            "variable": "city",
            "description": "Parsed job city.",
        },
        {
            "variable": "state",
            "description": "Parsed job state.",
        },
        {
            "variable": "skills_text",
            "description": "Extracted skills converted into readable text.",
        },
        {
            "variable": "role_match",
            "description": "Whether the title matched the Data Scientist pathway keywords.",
        },
        {
            "variable": "staffing_flag",
            "description": "Whether the posting appeared to be from a staffing agency.",
        },
    ]
)

dictionary_file = PROCESSED_DIR / "data_dictionary.csv"

data_dictionary.to_csv(dictionary_file, index=False)

display(data_dictionary)

print("Saved data dictionary to:", dictionary_file)

,variable,description
0,job_id,Unique API identifier for each job posting.
1,title,Original job title from the posting.
2,company_name,Employer name from the posting.
3,posted_at,"Posting date, when available."
4,naics_code,API-provided NAICS industry code.
5,naics_name,API-provided NAICS industry label.
6,remote_status,"Remote, hybrid, onsite, or unknown work arrangement."
7,annual_salary_min,Normalized lower annual-salary amount.
8,annual_salary_max,Normalized upper annual-salary amount.
9,annual_salary_midpoint,Average of the annual minimum and maximum salary.


Saved data dictionary to: /home/ubuntu/ad688-career-evaluation-group9/data/processed/data_dictionary.csv


In [32]:
facet_response = requests.get(
    f"{API_URL}/facets/",
    headers=HEADERS,
    params={
        "naics": "5132",
        "q": "data scientist",
    },
    timeout=180,
)

facet_response.raise_for_status()

facet_payload = facet_response.json()

print("Facet metadata:")
print(facet_payload.get("meta", {}))

print("\nApplied filters:")
print(facet_payload.get("filters", {}))

print("\nAvailable facet counts:")
print(facet_payload)

ReadTimeout: HTTPSConnectionPool(host='met-employability-services.azurewebsites.net', port=443): Read timed out. (read timeout=180)